# 00 · Smoke test: SmolVLA + LIBERO on Kaggle

**Before running:** Accelerator **GPU T4 x2**, Internet **On**. Then *Run All*.
Every step appends to `/kaggle/working/smoke_report.txt`; the last cell prints it.

In [ ]:
%%bash
echo "=== ENV ===" >> /kaggle/working/smoke_report.txt
nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv | tee -a /kaggle/working/smoke_report.txt
echo "CPUs: $(nproc)" | tee -a /kaggle/working/smoke_report.txt
free -g | head -2 | tee -a /kaggle/working/smoke_report.txt

In [ ]:
%%bash
# System libraries for headless MuJoCo rendering and video decoding
apt-get update -qq > /dev/null 2>&1
apt-get install -y -qq libegl1 libgl1 libglvnd0 libosmesa6 libosmesa6-dev ffmpeg cmake build-essential > /dev/null 2>&1 \
  && echo "apt: ok" | tee -a /kaggle/working/smoke_report.txt || { echo "apt: FAILED" | tee -a /kaggle/working/smoke_report.txt; true; }
# LeRobot needs Python >= 3.12; build an isolated venv with uv
pip install -q -U uv
uv python install 3.12 > /dev/null 2>&1
rm -rf /tmp/venv && uv venv /tmp/venv --python 3.12 > /dev/null 2>&1
/tmp/venv/bin/python --version > /dev/null 2>&1 \
  && echo "venv: ok ($(/tmp/venv/bin/python --version))" | tee -a /kaggle/working/smoke_report.txt || { echo "venv: FAILED" | tee -a /kaggle/working/smoke_report.txt; true; }

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
LOG=/kaggle/working/install.log
cd /tmp && rm -rf lerobot && git clone -q https://github.com/huggingface/lerobot.git && cd lerobot && git checkout -q ba83e4cf43737e2665be0aafcb5f513b83c7d0f3
echo "lerobot commit: $(git rev-parse HEAD)" | tee -a /kaggle/working/smoke_report.txt
unset UV_PYTHON UV_PROJECT_ENVIRONMENT PIP_TARGET
# Target the venv explicitly: Kaggle's environment otherwise sends uv to the system Python
uv pip install --python /tmp/venv/bin/python -e ".[smolvla,libero]" > $LOG 2>&1
RC=$?
echo "install exit: $RC ($(head -1 $LOG))" | tee -a /kaggle/working/smoke_report.txt
if [ $RC -ne 0 ]; then tail -40 $LOG | tee -a /kaggle/working/smoke_report.txt; true; fi
python - <<'EOF' 2>&1 | tee -a /kaggle/working/smoke_report.txt
import importlib, torch
for m in ["torch", "lerobot", "transformers", "mujoco", "libero"]:
    mod = importlib.import_module(m)
    print(f"import {m}: ok {getattr(mod, '__version__', '')}")
print("cuda:", torch.version.cuda, "|", torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
EOF
[ ${PIPESTATUS[0]} -eq 0 ] || true
# LIBERO asks for a dataset path on first import; answer "N" to keep defaults
(yes N | python -c "from libero.libero import benchmark; print('libero:', len(benchmark.get_benchmark_dict()), 'suites')") 2>&1 | tail -1 | tee -a /kaggle/working/smoke_report.txt
# Kaggle ships its own /usr/local/bin/lerobot-* that shadow the venv's; every call below uses `/tmp/venv/bin/python -m ...`
echo "lerobot-eval on PATH: $(which -a lerobot-eval | tr '\n' ' ')" | tee -a /kaggle/working/smoke_report.txt
/tmp/venv/bin/python -c "import lerobot, libero, mujoco" && touch /tmp/setup_ok && echo "setup: ok" | tee -a /kaggle/working/smoke_report.txt

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
python -c "import torch" 2>/dev/null || { echo "SKIP: install failed, see install.log" | tee -a /kaggle/working/smoke_report.txt; exit 0; }
# Try GPU (EGL) rendering first, fall back to CPU (OSMesa)
for GL in egl osmesa; do
  if MUJOCO_GL=$GL PYOPENGL_PLATFORM=$GL python - <<'EOF'
import mujoco
m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><light pos="0 0 1"/><geom type="sphere" size=".1"/></worldbody></mujoco>')
d = mujoco.MjData(m); mujoco.mj_forward(m, d)
r = mujoco.Renderer(m, 128, 128); r.update_scene(d); img = r.render()
assert img.mean() > 0
EOF
  then echo "render backend: $GL" | tee -a /kaggle/working/smoke_report.txt; echo $GL > /tmp/gl.txt; break
  else echo "render $GL: FAILED" | tee -a /kaggle/working/smoke_report.txt; fi
done

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
python -c "import torch" 2>/dev/null || { echo "SKIP: install failed, see install.log" | tee -a /kaggle/working/smoke_report.txt; exit 0; }
GL=$(cat /tmp/gl.txt 2>/dev/null || echo egl); export MUJOCO_GL=$GL PYOPENGL_PLATFORM=$GL
nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -i 0 -lms 1000 > /tmp/mem_eval.txt &
MON=$!
START=$(date +%s)
/tmp/venv/bin/python -m lerobot.scripts.lerobot_eval \
  --policy.path=HuggingFaceVLA/smolvla_libero \
  --env.type=libero --env.task=libero_goal '--env.task_ids=[0]' \
  --eval.n_episodes=2 --eval.batch_size=1 \
  --policy.device=cuda \
  --output_dir=/kaggle/working/smoke_eval 2>&1 | tail -25
EXIT=${PIPESTATUS[0]}
kill $MON
echo "=== EVAL (2 episodes, libero_goal task 0) exit=$EXIT seconds=$(( $(date +%s) - START )) peakMiB=$(sort -n /tmp/mem_eval.txt | tail -1)" | tee -a /kaggle/working/smoke_report.txt

In [ ]:
%%bash
source /tmp/venv/bin/activate; unset PYTHONPATH UV_SYSTEM_PYTHON; export MPLBACKEND=Agg
python -c "import torch" 2>/dev/null || { echo "SKIP: install failed, see install.log" | tee -a /kaggle/working/smoke_report.txt; exit 0; }
nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -i 0 -lms 1000 > /tmp/mem_train.txt &
MON=$!
START=$(date +%s)
/tmp/venv/bin/python -m lerobot.scripts.lerobot_train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=lerobot/libero \
  '--rename_map={"observation.images.image": "observation.images.camera1", "observation.images.image2": "observation.images.camera2"}' \
  --batch_size=8 --steps=50 --log_freq=10 --save_freq=100000 \
  --policy.device=cuda --policy.push_to_hub=false \
  --output_dir=/tmp/smoke_train 2>&1 | tail -30
EXIT=${PIPESTATUS[0]}
kill $MON
echo "=== TRAIN (50 steps, bs 8, incl. dataset download) exit=$EXIT seconds=$(( $(date +%s) - START )) peakMiB=$(sort -n /tmp/mem_train.txt | tail -1)" | tee -a /kaggle/working/smoke_report.txt

In [ ]:
print(open("/kaggle/working/smoke_report.txt").read())